# Preprocesamiento Textual

In [1]:
!pip install PyMuPDF pandas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import pandas as pd
import re

### Toma de datos de los PDF

In [3]:
# Ruta raíz donde están los apuntes PDF
root = r"Documentos_de_Inteligencia_artificial_GR 2\Apuntadores"

# Lista vacía para guardar todos los resultados
documentos = []

# Recorremos todos los archivos de la carpeta
for archivo in os.listdir(root):
    # Nos aseguramos de trabajar solo con archivos que terminen en .pdf
    if archivo.lower().endswith(".pdf"):
        
        # Creamos la ruta completa del archivo
        ruta_pdf = os.path.join(root, archivo)
        
        # Abrimos el PDF con PyMuPDF
        with fitz.open(ruta_pdf) as pdf:
            texto_total = ""
            
            # Recorremos cada página y extraemos su texto
            # Usamos flags para mejor extracción de texto con Unicode
            for pagina in pdf:
                # flags=0 es el modo más simple y limpio
                # También podemos usar flags=fitz.TEXT_PRESERVE_WHITESPACE
                texto_total += pagina.get_text("text", flags=0) + "\n"
        
        # Guardamos los datos extraídos en una estructura temporal (diccionario)
        documentos.append({
            "nombre_archivo": archivo,
            "ruta": ruta_pdf,
            "texto": texto_total.strip()  # Quitamos espacios innecesarios al inicio y final
        })

# Convertimos la lista de documentos en una tabla (DataFrame) para trabajar más fácil
df_docs = pd.DataFrame(documentos)

# Mostramos las primeras filas para verificar que funcionó
df_docs.head()


,nombre_archivo,ruta,texto
0,10_SEMANA_AI_20251007_1-222887296.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...
1,10_SEMANA_AI_20251007_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Redes Neuronales Convolucionales y\nBackpropag...
2,10_SEMANA_AI_20251009_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes de clase #2\nLuis Felipe Calderón Pére...
3,11_Semana_AI_20251014_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...
4,11_Semana_AI_20251014_2.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,"Inteligencia Artificial\nApuntes Semana 11, Cl..."


### Extracción de metadatos

In [4]:
import unicodedata

# Expresión regular para capturar datos del nombre del archivo
# Formato esperado: #semana_SEMANA_AI_yyyyddmm_#apunte.pdf
# Ejemplo: 10_SEMANA_AI_20251007_1.pdf
patron_nombre = r"(\d+)_semana_ai_(\d{4})(\d{2})(\d{2})_(\d+)\.pdf"

# Lista donde guardaremos los nuevos registros enriquecidos
datos_finales = []

# Definir lista de autores conocidos (estudiantes) - YA SIN TILDES
autores_conocidos = [
    "acuna lopez rodolfo david",
    "araya ortega fabian enrique",
    "benavides villegas luis fernando",
    "brenes martinez david",
    "brenes reyes fernando daniel",
    "brenes torres isaac david",
    "calderon perez luis felipe",
    "campos cerdas mauricio alonso",
    "carranza jimenez kevin josue",
    "diaz barboza fabian esteban",
    "espinoza aguilar dario",
    "gomez brenes gerardo alberto",
    "gonzalez sanchez luis alfredo",
    "jimenez salgado joselyn priscilla",
    "jimenez valverde juan diego",
    "murillo campos ian david",
    "naranjo masis alex steven",
    "oporta perez gianmarco",
    "quesada rodriguez jose pablo",
    "quesada sanchez mariana",
    "rodriguez camacho kendall andres",
    "rodriguez cano juan pablo",
    "rojas chacon sahid edgardo",
    "rojas obando nelson armando",
    "rojas rojas javier alonso",
    "sanchez araya brandon emmanuel",
    "sanchez rojas andres",
    "urena bermudez andrey",
    "varela venegas julio josue",
    "vargas solis rafael guillermo",
    "vasquez concepcion ashley lizeth",
    "vega suazo eder jose"
]

# Crear variaciones para mejor detección (apellidos completos) - SIN TILDES
apellidos_autores = [
    "acuna", "araya", "benavides", "brenes", "calderon", "campos",
    "carranza", "diaz", "espinoza", "gomez", "gonzalez", "jimenez",
    "murillo", "naranjo", "oporta", "quesada", "rodriguez", "rojas",
    "sanchez", "urena", "varela", "vargas", "vasquez", "vega"
]

for i, fila in df_docs.iterrows():
    nombre = fila["nombre_archivo"]
    texto = fila["texto"]

    # ------------------------------------------------------
    # 1. EXTRAER DATOS DEL NOMBRE DEL ARCHIVO
    # ------------------------------------------------------
    match = re.search(patron_nombre, nombre.lower())
    if match:
        semana = int(match.group(1))
        anio = int(match.group(2))
        dia = int(match.group(3))
        mes = int(match.group(4))
        apunte = int(match.group(5))
        fecha = f"{anio}-{mes:02d}-{dia:02d}"
    else:
        semana = None
        fecha = None
        apunte = None

    # ------------------------------------------------------
    # 2. EXTRAER TÍTULO Y AUTOR
    # ------------------------------------------------------
    lineas = texto.splitlines()
    titulo = lineas[0].strip() if lineas else None

    autor = None
    email = None

    # Recorremos línea por línea el texto extraído del PDF
    for linea in lineas[:15]:  # Solo las primeras 15 líneas (encabezado)
        linea_limpia = linea.strip().lower()
        
        # Buscar correo electrónico
        if "@estudiantec.cr" in linea_limpia:
            email = linea.strip()
            
        # Detectar autor: buscar coincidencias con apellidos conocidos
        # y que contenga al menos 2 palabras (nombre y apellido)
        if not autor and len(linea_limpia.split()) >= 2:
            # Verificar si contiene apellidos de estudiantes
            for apellido in apellidos_autores:
                if apellido in linea_limpia:
                    # Verificar que no contenga palabras del profesor
                    if "pacheco" not in linea_limpia and "portuguez" not in linea_limpia:
                        # Verificar que no sea el título ni otras líneas de metadata
                        if "abstract" not in linea_limpia and "index terms" not in linea_limpia:
                            autor = linea.strip()
                            break

    # ------------------------------------------------------
    # 3. EXTRAER ABSTRACT (RESUMEN)
    # ------------------------------------------------------
    abstract = None
    abstract_match = re.search(r"Abstract[—\-\s]+(.*?)(?:Index Terms|I\.|$)", texto, re.DOTALL | re.IGNORECASE)
    if abstract_match:
        abstract = abstract_match.group(1).strip()

    # ------------------------------------------------------
    # 4. GUARDAR DATOS ORGANIZADOS
    # ------------------------------------------------------
    datos_finales.append({
        "semana": semana,
        "fecha": fecha,
        "apunte": apunte,
        "titulo": titulo,
        "autor": autor,
        "email": email,
        "abstract": abstract,
        "texto": texto.strip(),
        "fuente": nombre
    })

# Crear un nuevo DataFrame con los metadatos enriquecidos
df_docs_full = pd.DataFrame(datos_finales)

# Mostrar una vista previa de los primeros registros
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,Apuntes IA Clase 7/10,Gianmarco Oporta P´erez,gooporta@estudiantec.cr,El presente documento recopila los apuntes de ...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...,10_SEMANA_AI_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,Redes Neuronales Convolucionales y,None,rodolfoide69@estudiantec.cr,En este documento podr´a encontrar informaci´o...,Redes Neuronales Convolucionales y\nBackpropag...,10_SEMANA_AI_20251007_1.pdf
2,10.0,2025-09-10,1.0,Apuntes de clase #2,None,None,None,Apuntes de clase #2\nLuis Felipe Calderón Pére...,10_SEMANA_AI_20251009_1.pdf
3,11.0,2025-14-10,1.0,Apuntes IA Clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,Este documento resume los conceptos clave vist...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...,11_Semana_AI_20251014_1.pdf
4,11.0,2025-14-10,2.0,Inteligencia Artificial,Luis Fernando Benavides Villegas,lubenavides@estudiantec.cr,Este documento recopila los apuntes de la clas...,"Inteligencia Artificial\nApuntes Semana 11, Cl...",11_Semana_AI_20251014_2.pdf


### Limpieza de datos

In [5]:
import unicodedata

def eliminar_tildes(texto):
    """
    Elimina tildes y acentos manejando:
    1. Unicode compuesto (é)
    2. Unicode descompuesto (e + ´)
    3. Caracteres literales de acento (´, `, ¨, ^, ~)
    4. Caracteres especiales resultantes (ı, ȷ, etc.)
    """
    if not texto or not isinstance(texto, str):
        return texto
    
    # Paso 1: Eliminar caracteres literales de acento que aparecen solos
    acentos_literales = ['´', '`', '¨', '^', '~', '¯', '˜', '¸', 'ˆ', '˙', '˚', '˝']
    for acento in acentos_literales:
        texto = texto.replace(acento, '')
    
    # Paso 2: Normalizar a NFD (descomponer caracteres acentuados)
    texto_nfd = unicodedata.normalize('NFD', texto)
    
    # Paso 3: Filtrar marcas diacríticas (categoría Mn)
    texto_sin_tildes = ''.join(
        char for char in texto_nfd 
        if unicodedata.category(char) != 'Mn'
    )
    
    # Paso 4: Normalizar a NFC (recomponer)
    texto_final = unicodedata.normalize('NFC', texto_sin_tildes)
    
    # Paso 5: Reemplazar caracteres especiales que quedan
    # ı (dotless i) -> i
    # ȷ (dotless j) -> j
    # ø (o con barra) -> o
    # etc.
    reemplazos_especiales = {
        'ı': 'i',  # Latin small letter dotless i
        'ȷ': 'j',  # Latin small letter dotless j
        'ø': 'o',  # Latin small letter o with stroke
        'Ø': 'O',  # Latin capital letter O with stroke
        'ł': 'l',  # Latin small letter l with stroke
        'Ł': 'L',  # Latin capital letter L with stroke
        'ð': 'd',  # Latin small letter eth
        'Ð': 'D',  # Latin capital letter ETH
        'þ': 'th', # Latin small letter thorn
        'Þ': 'Th', # Latin capital letter THORN
        'ß': 'ss', # Latin small letter sharp s
    }
    
    for especial, normal in reemplazos_especiales.items():
        texto_final = texto_final.replace(especial, normal)
    
    return texto_final

def limpiar_campo_simple(texto):
    """Limpia campos individuales (autor, email, título)"""
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios
    texto = " ".join(texto.split())
    
    return texto

def limpiar_texto(texto):
    """Limpia campos de texto largo (abstract, texto completo)"""
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios y saltos de línea
    texto = " ".join(texto.split())
    
    return texto

def validar_autor(autor):
    """Valida si el autor detectado está en la lista de autores conocidos"""
    if not autor or not isinstance(autor, str):
        return autor
    
    autor_limpio = limpiar_campo_simple(autor)
    
    # Verificar si está en la lista de autores conocidos
    for nombre_conocido in autores_conocidos:
        if nombre_conocido in autor_limpio or autor_limpio in nombre_conocido:
            return autor
    
    return autor  # Si no se encuentra, mantener el original

# Aplicar limpieza a todos los campos del DataFrame
df_docs_full["titulo"] = df_docs_full["titulo"].apply(limpiar_campo_simple)
df_docs_full["autor"] = df_docs_full["autor"].apply(lambda x: validar_autor(x)).apply(limpiar_campo_simple)
df_docs_full["email"] = df_docs_full["email"].apply(limpiar_campo_simple)
df_docs_full["abstract"] = df_docs_full["abstract"].apply(limpiar_texto)
df_docs_full["texto"] = df_docs_full["texto"].apply(limpiar_texto)
df_docs_full["fuente"] = df_docs_full["fuente"].apply(limpiar_campo_simple)

# Mostrar todas las columnas (incluyendo texto limpio)
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,apuntes ia clase 7/10,gianmarco oporta perez,gooporta@estudiantec.cr,el presente documento recopila los apuntes de ...,apuntes ia clase 7/10 gianmarco oporta perez i...,10_semana_ai_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,redes neuronales convolucionales y,None,rodolfoide69@estudiantec.cr,en este documento podra encontrar informacion ...,redes neuronales convolucionales y backpropaga...,10_semana_ai_20251007_1.pdf
2,10.0,2025-09-10,1.0,apuntes de clase #2,None,None,None,apuntes de clase #2 luis felipe calderon perez...,10_semana_ai_20251009_1.pdf
3,11.0,2025-14-10,1.0,apuntes ia clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,este documento resume los conceptos clave vist...,apuntes ia clase 14/10/2025 juan jimenez valve...,11_semana_ai_20251014_1.pdf
4,11.0,2025-14-10,2.0,inteligencia artificial,luis fernando benavides villegas,lubenavides@estudiantec.cr,este documento recopila los apuntes de la clas...,"inteligencia artificial apuntes semana 11, cla...",11_semana_ai_20251014_2.pdf


### congelar dataframe en un .parquet

### Guardar DataFrame procesado

In [6]:
!pip install -U pandas pyarrow fastparquet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Guardar el DataFrame limpio en formato Parquet
# Esto permite versionarlo y cargarlo rápidamente después
output_path = "df_docs_procesado.parquet"

try:
    # Intentar guardar con pyarrow
    df_docs_full.to_parquet(output_path, engine='pyarrow', compression='snappy')
    print(f"DataFrame guardado exitosamente en: {output_path}")
except Exception as e:
    # Si falla, usar pickle como alternativa
    print(f"Error con parquet: {e}")
    print("Guardando con pickle como alternativa...")
    output_path = "df_docs_procesado.pkl"
    df_docs_full.to_pickle(output_path)
    print(f"DataFrame guardado exitosamente en: {output_path}")

print(f"Total de documentos: {len(df_docs_full)}")
print(f"Columnas guardadas: {list(df_docs_full.columns)}")

DataFrame guardado exitosamente en: df_docs_procesado.parquet
Total de documentos: 46
Columnas guardadas: ['semana', 'fecha', 'apunte', 'titulo', 'autor', 'email', 'abstract', 'texto', 'fuente']


In [8]:
import os
import pandas as pd

def guardar_parquet_con_fallback(df, ruta):
    try:
        import pyarrow as pa  # asegúrate de importarlo aquí
        try:
            pa.unregister_extension_type("pandas.period")
        except Exception:
            pass
        df.to_parquet(ruta, index=False, engine="pyarrow", compression="snappy")
        print(f"Guardado con pyarrow en: {ruta}")
    except Exception as e:
        print("pyarrow falló:", e, "\n→ Intentando con fastparquet…")
        df.to_parquet(ruta, index=False, engine="fastparquet", compression="snappy")
        print(f"Guardado con fastparquet en: {ruta}")

carpeta_salida = "data"
os.makedirs(carpeta_salida, exist_ok=True)
ruta_parquet = os.path.join(carpeta_salida, "apuntes_clean_v1.parquet")

guardar_parquet_con_fallback(df_docs_full, ruta_parquet)

Guardado con pyarrow en: data\apuntes_clean_v1.parquet


### Tecnicas de segmentación

#### Fixed-size Chunking with sliding window

In [9]:
# Fixed-size chunking con ventana deslizante (caracteres)
# Contrato:
# - Entrada: dataset limpio cargado desde data/*.parquet (sin modificar el DF original)
# - Salida: df_chunks_sliding con un chunk por fila, preservando metadatos útiles
# - Regla: stride = chunk_size - overlap

import os
import pandas as pd
from typing import List, Dict

# 1) Loader robusto: lee el dataset "congelado" desde /data
#    Intenta pyarrow -> fastparquet -> pickle de respaldo

def load_dataset():
    candidatos = [
        os.path.join("data", "apuntes_clean_v1.parquet"),
        os.path.join("data", "df_docs_procesado.parquet"),
    ]
    for ruta in candidatos:
        if os.path.exists(ruta):
            # intentar pyarrow, luego fastparquet
            try:
                return pd.read_parquet(ruta, engine="pyarrow")
            except Exception:
                try:
                    return pd.read_parquet(ruta, engine="fastparquet")
                except Exception:
                    pass
    # Fallback a pickle si existe
    pkl = os.path.join("data", "df_docs_procesado.pkl")
    if os.path.exists(pkl):
        return pd.read_pickle(pkl)
    # Último recurso: si existe en la raíz (caso previo)
    if os.path.exists("df_docs_procesado.pkl"):
        return pd.read_pickle("df_docs_procesado.pkl")
    raise FileNotFoundError(
        "No se encontró el dataset en data/*.parquet ni el pickle de respaldo."
    )


def chunk_text_sliding_window(texto: str, chunk_size: int = 1000, overlap: int = 200) -> List[Dict]:
    """
    Divide un texto en trozos (chunks) de tamaño fijo usando una ventana deslizante.
    - chunk_size: tamaño máximo del chunk en caracteres.
    - overlap: cantidad de caracteres que se solapan entre chunks consecutivos.
    Retorna una lista de dicts con: start, end, length, chunk, chunk_index.
    """
    if not isinstance(texto, str) or not texto:
        return []

    if chunk_size <= 0:
        raise ValueError("chunk_size debe ser > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap debe estar en [0, chunk_size)")

    stride = chunk_size - overlap
    chunks: List[Dict] = []

    n = len(texto)
    idx = 0
    start = 0
    while start < n:
        end = min(start + chunk_size, n)
        fragmento = texto[start:end]
        if fragmento.strip():
            chunks.append({
                "chunk_index": idx,
                "start": start,
                "end": end,
                "length": end - start,
                "chunk": fragmento,
            })
            idx += 1
        if end == n:
            break
        start += stride

    return chunks

# 2) Cargar dataset desde /data sin tocar el DF original
pd.set_option('display.max_colwidth', None)
df_src = load_dataset()  # columnas esperadas: semana, fecha, apunte, titulo, autor, email, abstract, texto, fuente

# 3) Aplicar segmentación a todo el dataset y construir un DataFrame de chunks
registros = []
for row in df_src.itertuples(index=False):
    texto_doc = getattr(row, "texto")
    fuente = getattr(row, "fuente")
    for ch in chunk_text_sliding_window(texto_doc, chunk_size=1000, overlap=200):
        registros.append({
            "fuente": fuente,
            "semana": getattr(row, "semana", None),
            "fecha": getattr(row, "fecha", None),
            "apunte": getattr(row, "apunte", None),
            "titulo": getattr(row, "titulo", None),
            "autor": getattr(row, "autor", None),
            "chunk_index": ch["chunk_index"],
            "start": ch["start"],
            "end": ch["end"],
            "length": ch["length"],
            "chunk": ch["chunk"],
        })

df_chunks_sliding = pd.DataFrame(registros).sort_values(["fuente", "chunk_index"]).reset_index(drop=True)

# 4) Resumen y vista
num_docs = df_src.shape[0]
num_chunks = df_chunks_sliding.shape[0]
print(f" Documentos: {num_docs} | Chunks generados: {num_chunks} | Promedio por doc: {num_chunks / max(num_docs,1):.2f}")

df_chunks_sliding.head(10)

# 5) (Opcional) Guardar a disco para versionar resultados de segmentación
try:
    carpeta_salida = "data"
    os.makedirs(carpeta_salida, exist_ok=True)
    ruta_chunks = os.path.join(carpeta_salida, "chunks_sliding_v1.parquet")
    # Reusar el helper si existe en el notebook, sino guardar con parquet simple
    try:
        guardar_parquet_con_fallback(df_chunks_sliding, ruta_chunks)
    except NameError:
        # Si no está definida la función auxiliar, intentamos parquet directo con fallback simple
        try:
            df_chunks_sliding.to_parquet(ruta_chunks, engine="pyarrow", index=False, compression="snappy")
            print(f"Chunks guardados en: {ruta_chunks}")
        except Exception:
            alt = os.path.join(carpeta_salida, "chunks_sliding_v1.pkl")
            df_chunks_sliding.to_pickle(alt)
            print(f"Chunks guardados (pickle) en: {alt}")
except Exception as e:
    print("No se guardó parquet de chunks (opcional):", e)

 Documentos: 46 | Chunks generados: 620 | Promedio por doc: 13.48
Guardado con pyarrow en: data\chunks_sliding_v1.parquet


#### Recursive chunking (Recursive Text Splitter)

Este método intenta mantener límites “naturales” del texto antes de forzarlo a tamaños fijos:
- Empieza dividiendo por separadores más fuertes a más débiles (párrafos → líneas → oraciones → signos → espacios).
- Si un fragmento sigue siendo largo, vuelve a dividirlo recursivamente con el siguiente separador.
- Al final, recombina piezas en chunks de tamaño objetivo con un solapamiento (overlap) definido para preservar contexto entre chunks.

Ventajas:
- Chunks más “semánticos” (menos cortes bruscos a mitad de ideas).
- Control fino del tamaño y del solapamiento.

Parámetros típicos:
- chunk_size: 1000 caracteres
- overlap: 200 caracteres
- separadores: ["\n\n", "\n", ". ", "; ", ", ", " "] (de fuerte → débil)

In [10]:
# Segmentación recursiva desde data/apuntes_clean_v1.parquet y guardado en data/chunks_recursive_v1.parquet
import os
import re
import pandas as pd
from typing import List

# Reutilizamos el loader robusto (pyarrow -> fastparquet -> pickle)
def load_dataset():
    rutas_intentos = [
        os.path.join('data', 'apuntes_clean_v1.parquet'),
        'apuntes_clean_v1.parquet',
    ]
    ultimo_error = None
    for ruta in rutas_intentos:
        if os.path.exists(ruta):
            try:
                try:
                    return pd.read_parquet(ruta, engine='pyarrow')
                except Exception as e1:
                    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
                    try:
                        return pd.read_parquet(ruta, engine='fastparquet')
                    except Exception as e2:
                        ultimo_error = (e1, e2)
                        print(f"fastparquet también falló: {e2}")
            except Exception as e:
                ultimo_error = e
    # Fallback a pickle si existe
    for ruta in ['df_docs_procesado.pkl', os.path.join('data','df_docs_procesado.pkl')]:
        if os.path.exists(ruta):
            print("Cargando dataset desde pickle de respaldo…")
            return pd.read_pickle(ruta)
    raise RuntimeError(f"No se pudo cargar el dataset. Último error: {ultimo_error}")

# Utilidades: normalizar espacios y asegurar strings
_ws_re = re.compile(r"\s+")

def _s(text):
    if text is None:
        return ""
    return str(text)

# Split recursivo por una lista jerárquica de separadores
# separadores: de fuerte → débil (p. ej. párrafos → líneas → oraciones → espacios)

def recursive_split(text: str, separators: List[str], max_len: int) -> List[str]:
    text = _s(text).strip()
    if not text:
        return []
    # Caso base: si ya es corto, devolver tal cual
    if len(text) <= max_len:
        return [text]

    if not separators:
        # Sin separadores disponibles: forzar corte duro
        return [text[i:i+max_len] for i in range(0, len(text), max_len)]

    sep = separators[0]
    rest = separators[1:]

    # Dividir por el separador actual; si el separador es espacio simple, usar split(' ')
    if sep == ' ':
        parts = text.split(' ')
        glue = ' '
    else:
        parts = text.split(sep)
        glue = sep

    # Si el separador no generó cortes (texto sin ese separador), avanzar al siguiente
    if len(parts) == 1:
        return recursive_split(text, rest, max_len)

    # Repartir recursivamente cada parte si excede tamaño
    chunks = []
    current = []
    current_len = 0

    def flush_current():
        nonlocal current, current_len
        if current:
            combined = glue.join(current).strip()
            if combined:
                if len(combined) <= max_len:
                    chunks.append(combined)
                else:
                    # Aún grande: bajar a siguiente separador
                    chunks.extend(recursive_split(combined, rest, max_len))
        current = []
        current_len = 0

    for p in parts:
        piece = p.strip()
        if not piece:
            # Conserva separador donde sea útil; evitamos cadenas vacías consecutivas
            if current and glue:
                # Empuja un separador lógico en la recombinación
                current.append('')
            continue
        prospective_len = (current_len + (len(glue) if current else 0) + len(piece))
        if prospective_len <= max_len:
            # Aún cabe en el paquete actual
            if current:
                current.append(piece)
                current_len = prospective_len
            else:
                current = [piece]
                current_len = len(piece)
        else:
            # Cierra paquete actual y decide sobre piece
            flush_current()
            if len(piece) <= max_len:
                current = [piece]
                current_len = len(piece)
            else:
                # La pieza sola ya excede: desciende de nivel
                chunks.extend(recursive_split(piece, rest, max_len))
                current = []
                current_len = 0

    flush_current()
    return chunks

# Recombina listas de trozos cortos en ventanas con solapamiento, manteniendo tamaño objetivo

def pack_with_overlap(frags: List[str], chunk_size: int, overlap: int) -> List[str]:
    if not frags:
        return []
    # Normaliza y filtra vacíos
    norm = []
    for f in frags:
        s = _ws_re.sub(' ', _s(f)).strip()
        if s:
            norm.append(s)
    if not norm:
        return []

    chunks = []
    buf = []
    buf_len = 0

    def emit():
        nonlocal buf, buf_len
        if buf:
            out = ' '.join(buf).strip()
            if out:
                chunks.append(out)
        buf = []
        buf_len = 0

    for frag in norm:
        if buf_len == 0:
            buf.append(frag)
            buf_len = len(frag)
            continue
        prospective = buf_len + 1 + len(frag)  # +1 por espacio
        if prospective <= chunk_size:
            buf.append(frag)
            buf_len = prospective
        else:
            # Emite actual y aplica solapamiento aproximado por palabras
            emit()
            if overlap > 0 and chunks:
                tail = chunks[-1]
                # Toma ~overlap caracteres desde el final, cortado por palabras
                # Luego extrae últimas palabras para reconstruir contexto
                if len(tail) > overlap:
                    tail_ctx = tail[-overlap:]
                    # Evita iniciar en medio de palabra
                    tail_ctx = tail_ctx[tail_ctx.find(' ')+1:] if ' ' in tail_ctx else tail_ctx
                    if tail_ctx:
                        buf = [tail_ctx]
                        buf_len = len(tail_ctx)
                    else:
                        buf = []
                        buf_len = 0
                else:
                    buf = [tail]
                    buf_len = len(tail)
            else:
                buf = []
                buf_len = 0
            # Añade el fragmento actual (puede iniciar nuevo buffer)
            if buf_len == 0:
                buf = [frag]
                buf_len = len(frag)
            else:
                prospective = buf_len + 1 + len(frag)
                if prospective <= chunk_size:
                    buf.append(frag)
                    buf_len = prospective
                else:
                    emit()
                    buf = [frag]
                    buf_len = len(frag)

    emit()
    return chunks

# Pipeline: carga → split recursivo → empaquetado con solapamiento → DataFrame y guardado

df_docs_frozen = load_dataset()
assert 'texto' in df_docs_frozen.columns, "El dataset cargado debe contener la columna 'texto'"

separators = ["\n\n", "\n", ". ", "; ", ", ", " "]
chunk_size = 1000
overlap = 200

registros = []
for idx, row in df_docs_frozen.iterrows():
    texto = _s(row.get('texto', ''))
    if not texto.strip():
        continue
    # 1) Split recursivo por jerarquía de separadores
    frags = recursive_split(texto, separators, max_len=chunk_size)
    # 2) Empaquetar con solapamiento para uniformar tamaño objetivo
    chunks = pack_with_overlap(frags, chunk_size=chunk_size, overlap=overlap)

    for c_idx, c in enumerate(chunks):
        registros.append({
            'fuente': row.get('fuente'),
            'semana': row.get('semana'),
            'fecha': row.get('fecha'),
            'apunte': row.get('apunte'),
            'titulo': row.get('titulo'),
            'autor': row.get('autor'),
            'chunk_id': f"rec_{idx}_{c_idx}",
            'chunk_text': c,
            'chunk_len': len(c),
        })

# DataFrame de salida
chunks_recursive_df = pd.DataFrame(registros)
print(f" Documentos: {len(df_docs_frozen)} | Chunks generados (rec): {len(chunks_recursive_df)} | Promedio por doc: {len(chunks_recursive_df)/max(1,len(df_docs_frozen)):.2f}")
print(chunks_recursive_df['chunk_len'].describe().round(1))

# Guardado robusto en data/chunks_recursive_v1.parquet
os.makedirs('data', exist_ok=True)
salida = os.path.join('data', 'chunks_recursive_v1.parquet')
try:
    chunks_recursive_df.to_parquet(salida, engine='pyarrow', index=False)
    print(f"Guardado con pyarrow en: {salida}")
except Exception as e1:
    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
    try:
        chunks_recursive_df.to_parquet(salida, engine='fastparquet', index=False)
        print(f"Guardado con fastparquet en: {salida}")
    except Exception as e2:
        print(f"fastparquet también falló: {e2}\n→ Guardando respaldo pickle…")
        salida_pkl = os.path.join('data', 'chunks_recursive_v1.pkl')
        chunks_recursive_df.to_pickle(salida_pkl)
        print(f"Respaldo guardado en: {salida_pkl}")

 Documentos: 46 | Chunks generados (rec): 988 | Promedio por doc: 21.48
count     988.0
mean      595.1
std       361.9
min       180.0
25%       196.0
50%       836.5
75%       943.2
max      1000.0
Name: chunk_len, dtype: float64
Guardado con pyarrow en: data\chunks_recursive_v1.parquet


# Tokenización y Embeddings

In [11]:
!pip install -U pandas pyarrow numpy tqdm tenacity
!pip install -U openai python-dotenv


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Instalación de FAISS para Base de Datos Vectorial

FAISS (Facebook AI Similarity Search) es una biblioteca optimizada para búsqueda de similitud en espacios vectoriales de alta dimensión.

**Ventajas**:
- Búsqueda extremadamente rápida (millones de vectores/segundo)
- Bajo uso de memoria
- Soporte para CPU y GPU
- Ideal para prototipado y producción local

**Uso en este proyecto**:
- Crearemos 2 índices independientes (sliding y recursive)
- Cada índice almacenará ~600-1000 vectores de 1536 dimensiones
- Métrica: producto interno (equivalente a cosine similarity con vectores normalizados)

In [12]:
# Instalar FAISS (versión CPU) y tiktoken para tokenización de OpenAI
!pip install -q faiss-cpu tiktoken


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuración del Cliente OpenAI

Para generar embeddings con `text-embedding-3-small`, necesitamos:
1. Una API key válida de OpenAI
2. El cliente de la librería openai (v1.0+)
3. Un archivo `.env` con la configuración (más seguro)

**Configuración con archivo .env** (recomendado):
1. Edita el archivo `.env` en la raíz del proyecto
2. Reemplaza `sk-proj-tu-api-key-aqui` con tu API key real
3. El archivo `.env` ya está en `.gitignore` para proteger tus credenciales

**Formato del archivo .env**:
```
OPENAI_API_KEY=sk-proj-tu-api-key-real-aqui
```

La librería `python-dotenv` cargará automáticamente estas variables.

In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Verificar que la API key esté cargada
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró OPENAI_API_KEY.\n"
        "Edita el archivo .env en la raíz del proyecto y agrega tu API key:\n"
        "   OPENAI_API_KEY=sk-proj-tu-api-key-aqui\n"
    )

if api_key == "sk-proj-tu-api-key-aqui":
    raise ValueError(
        "Aún no has configurado tu API key real.\n"
        "Edita el archivo .env y reemplaza el placeholder con tu API key de OpenAI.\n"
        "Obtén tu key en: https://platform.openai.com/api-keys"
    )

# Inicializar cliente OpenAI
try:
    client = OpenAI(api_key=api_key)
    print("Cliente OpenAI inicializado correctamente")
    print(f"Modelo a usar: text-embedding-3-small (1536 dimensiones)")
    print(f"API key cargada desde .env: {api_key[:20]}...{api_key[-4:]}")
except Exception as e:
    print(f"Error al inicializar cliente OpenAI: {e}")
    raise

Cliente OpenAI inicializado correctamente
Modelo a usar: text-embedding-3-small (1536 dimensiones)
API key cargada desde .env: sk-proj-BSJOpTOHcMSe...rc0A


In [15]:
import numpy as np
from typing import List
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAI, RateLimitError, APITimeoutError, APIConnectionError

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError)),
    reraise=True
)
def _call_embedding_api(client: OpenAI, texts: List[str], model: str) -> List[List[float]]:
    """
    Llama a la API de OpenAI para generar embeddings con retry automático.
    
    Esta función interna maneja reintentos automáticos en caso de:
    - Rate limits (demasiadas requests)
    - Timeouts (servidor lento)
    - Connection errors (problemas de red)
    
    Args:
        client: Cliente OpenAI inicializado
        texts: Lista de textos (máx 100)
        model: Nombre del modelo de embeddings
        
    Returns:
        Lista de vectores (embeddings)
    """
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    return [item.embedding for item in response.data]


def generate_embeddings_batch(
    texts: List[str],
    client: OpenAI,
    model: str = "text-embedding-3-small",
    batch_size: int = 100,
    normalize: bool = True,
    show_progress: bool = True
) -> np.ndarray:
    """
    Genera embeddings para una lista de textos en lotes, con retry y progress bar.
    
    Args:
        texts: Lista de strings a convertir en embeddings
        client: Cliente OpenAI inicializado
        model: Modelo de OpenAI (default: "text-embedding-3-small", 1536 dims)
        batch_size: Cantidad de textos por request API (máx 100, default: 100)
        normalize: Normalizar vectores para cosine similarity (default: True)
        show_progress: Mostrar barra de progreso (default: True)
        
    Returns:
        numpy array de shape (n_texts, embedding_dim)
        
    Ejemplo:
        >>> embeddings = generate_embeddings_batch(
        ...     texts=["Hola mundo", "Python es genial"],
        ...     client=client
        ... )
        >>> embeddings.shape
        (2, 1536)
    """
    if not texts:
        return np.array([])
    
    # Filtrar textos vacíos y mantener índices originales
    valid_indices = [i for i, t in enumerate(texts) if t and str(t).strip()]
    valid_texts = [texts[i] for i in valid_indices]
    
    if not valid_texts:
        print("Advertencia: todos los textos están vacíos")
        return np.array([])
    
    print(f"Generando embeddings para {len(valid_texts)} textos...")
    print(f"   Modelo: {model}")
    print(f"   Batch size: {batch_size}")
    print(f"   Normalización: {'Sí' if normalize else 'No'}")
    
    all_embeddings = []
    
    # Procesar en lotes con progress bar
    iterator = range(0, len(valid_texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Generando embeddings", unit="batch")
    
    for i in iterator:
        batch = valid_texts[i:i + batch_size]
        
        try:
            # Llamar API con retry automático
            batch_embeddings = _call_embedding_api(client, batch, model)
            all_embeddings.extend(batch_embeddings)
            
        except Exception as e:
            print(f"\nError en batch {i//batch_size + 1}: {e}")
            raise
    
    # Convertir a numpy array
    embeddings_array = np.array(all_embeddings, dtype=np.float32)
    
    # Normalizar si se solicita (requerido para cosine similarity con FAISS)
    if normalize:
        norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
        embeddings_array = embeddings_array / norms
    
    print(f"Embeddings generados: shape {embeddings_array.shape}")
    print(f"   Rango de valores: [{embeddings_array.min():.4f}, {embeddings_array.max():.4f}]")
    
    # Si había textos vacíos, crear array completo con zeros en posiciones vacías
    if len(valid_indices) < len(texts):
        full_embeddings = np.zeros((len(texts), embeddings_array.shape[1]), dtype=np.float32)
        full_embeddings[valid_indices] = embeddings_array
        return full_embeddings
    
    return embeddings_array


# Test rápido de la función (comentar después de verificar)
print("\nTest de la función de embeddings:")
test_texts = ["inteligencia artificial", "aprendizaje automatico", "redes neuronales"]
test_embeddings = generate_embeddings_batch(test_texts, client, batch_size=3)
print(f"Test exitoso: {len(test_texts)} textos → shape {test_embeddings.shape}")


Test de la función de embeddings:
Generando embeddings para 3 textos...
   Modelo: text-embedding-3-small
   Batch size: 3
   Normalización: Sí


Generando embeddings:   0%|          | 0/1 [00:00<?, ?batch/s]

Generando embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.17batch/s]

Embeddings generados: shape (3, 1536)
   Rango de valores: [-0.1108, 0.1291]
Test exitoso: 3 textos → shape (3, 1536)


### Función para Generar Embeddings en Lotes

Esta función implementa las mejores prácticas para generar embeddings con la API de OpenAI:

**Características**:
- **Procesamiento en batch**: Procesa múltiples textos por request (hasta 100 por lote según límites de OpenAI)
- **Retry automático**: Usa `tenacity` para reintentar en caso de errores temporales (rate limits, timeouts)
- **Progress bar**: Muestra el progreso con `tqdm` para visualizar el avance
- **Normalización**: Los vectores se normalizan para usar cosine similarity con FAISS
- **Manejo de errores**: Captura y reporta errores específicos

**Parámetros**:
- `texts`: Lista de strings a convertir en embeddings
- `model`: Modelo de OpenAI (por defecto "text-embedding-3-small")
- `batch_size`: Cantidad de textos por request (máx 100)
- `normalize`: Si normalizar vectores para cosine similarity (recomendado para FAISS)

In [16]:
import faiss
import json
import os
import pandas as pd

# ============================================================================
# 1. CARGAR CHUNKS DE SLIDING WINDOW
# ============================================================================
print("=" * 80)
print("PROCESANDO CHUNKS SLIDING WINDOW")
print("=" * 80)

ruta_chunks_sliding = os.path.join("data", "chunks_sliding_v1.parquet")

try:
    df_sliding = pd.read_parquet(ruta_chunks_sliding, engine="fastparquet")
    print(f"Chunks cargados: {len(df_sliding)} registros")
    print(f"   Columnas: {list(df_sliding.columns)}")
except Exception as e:
    print(f" Error al cargar chunks: {e}")
    raise

# Extraer textos de la columna 'chunk'
texts_sliding = df_sliding["chunk"].tolist()
print(f" Textos a procesar: {len(texts_sliding)}")

# ============================================================================
# 2. GENERAR EMBEDDINGS
# ============================================================================
print("\n" + "=" * 80)
print("GENERANDO EMBEDDINGS")
print("=" * 80)

embeddings_sliding = generate_embeddings_batch(
    texts=texts_sliding,
    client=client,
    model="text-embedding-3-small",
    batch_size=100,
    normalize=True,
    show_progress=True
)

print(f"\n Embeddings generados: {embeddings_sliding.shape}")

# ============================================================================
# 3. CREAR ÍNDICE FAISS
# ============================================================================
print("\n" + "=" * 80)
print("CREANDO ÍNDICE FAISS")
print("=" * 80)

# Dimensión de los embeddings (1536 para text-embedding-3-small)
dimension = embeddings_sliding.shape[1]

# Crear índice FAISS con producto interno (equivale a cosine similarity con vectores normalizados)
index_sliding = faiss.IndexFlatIP(dimension)

# Agregar vectores al índice
index_sliding.add(embeddings_sliding)

print(f" Índice FAISS creado")
print(f"   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)")
print(f"   Dimensión: {index_sliding.d}")
print(f"   Vectores almacenados: {index_sliding.ntotal}")

# ============================================================================
# 4. GUARDAR ÍNDICE FAISS Y METADATA
# ============================================================================
print("\n" + "=" * 80)
print("GUARDANDO ÍNDICE Y METADATA")
print("=" * 80)

# Crear directorio vectordb
os.makedirs("vectordb", exist_ok=True)

# Guardar índice FAISS
index_path = os.path.join("vectordb", "faiss_index_sliding.bin")
faiss.write_index(index_sliding, index_path)
print(f" Índice guardado: {index_path}")

# Guardar metadata (información de los chunks para recuperación)
metadata_sliding = {
    "index_type": "IndexFlatIP",
    "dimension": dimension,
    "total_vectors": int(index_sliding.ntotal),
    "model": "text-embedding-3-small",
    "chunking_method": "fixed-size sliding window",
    "chunk_size": 1000,
    "overlap": 200,
    "normalized": True,
    "columns": list(df_sliding.columns),
    "created_at": pd.Timestamp.now().isoformat()
}

metadata_path = os.path.join("vectordb", "faiss_index_sliding_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_sliding, f, indent=2, ensure_ascii=False)
print(f" Metadata guardada: {metadata_path}")

# ============================================================================
# 5. GUARDAR BACKUP CON EMBEDDINGS
# ============================================================================
print("\n" + "=" * 80)
print("GUARDANDO BACKUP CON EMBEDDINGS")
print("=" * 80)

# Agregar embeddings al DataFrame como nueva columna
df_sliding_with_embeddings = df_sliding.copy()
df_sliding_with_embeddings["embedding"] = list(embeddings_sliding)

# Guardar con fallback
backup_path = os.path.join("data", "embeddings_sliding_v1.parquet")
try:
    df_sliding_with_embeddings.to_parquet(backup_path, engine="fastparquet", index=False)
    print(f" Backup guardado: {backup_path}")
except Exception as e:
    print(f"  Error con parquet: {e}")
    backup_path_pkl = os.path.join("data", "embeddings_sliding_v1.pkl")
    df_sliding_with_embeddings.to_pickle(backup_path_pkl)
    print(f" Backup guardado (pickle): {backup_path_pkl}")

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "=" * 80)
print(" PROCESAMIENTO COMPLETADO: CHUNKS SLIDING WINDOW")
print("=" * 80)
print(f" Chunks procesados: {len(df_sliding)}")
print(f" Embeddings generados: {embeddings_sliding.shape}")
print(f"  Índice FAISS: {index_sliding.ntotal} vectores")
print(f"\n Archivos generados:")
print(f"   - {index_path}")
print(f"   - {metadata_path}")
print(f"   - {backup_path if 'backup_path' in locals() else backup_path_pkl}")
print("=" * 80)

PROCESANDO CHUNKS SLIDING WINDOW
Chunks cargados: 620 registros
   Columnas: ['fuente', 'semana', 'fecha', 'apunte', 'titulo', 'autor', 'chunk_index', 'start', 'end', 'length', 'chunk']
 Textos a procesar: 620

GENERANDO EMBEDDINGS
Generando embeddings para 620 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí
Chunks cargados: 620 registros
   Columnas: ['fuente', 'semana', 'fecha', 'apunte', 'titulo', 'autor', 'chunk_index', 'start', 'end', 'length', 'chunk']
 Textos a procesar: 620

GENERANDO EMBEDDINGS
Generando embeddings para 620 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí


Generando embeddings: 100%|██████████| 7/7 [00:11<00:00,  1.65s/batch]

Embeddings generados: shape (620, 1536)
   Rango de valores: [-0.1570, 0.1853]

 Embeddings generados: (620, 1536)

CREANDO ÍNDICE FAISS
 Índice FAISS creado
   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)
   Dimensión: 1536
   Vectores almacenados: 620

GUARDANDO ÍNDICE Y METADATA
 Índice guardado: vectordb\faiss_index_sliding.bin
 Metadata guardada: vectordb\faiss_index_sliding_metadata.json

GUARDANDO BACKUP CON EMBEDDINGS
  Error con parquet: Can't infer object conversion type: 0      [-0.02525331, -0.00074718904, 0.017798211, -0.04450672, 0.012794564, -0.0021380242, 0.017954925, 0.03949188, -0.060491532, -0.015570636, 0.0037163561, -0.0012495128, -0.05776023, 0.00034700916, -0.03329049, -0.020596672, 0.002343711, -0.018772075, 0.0004222178, 0.028230874, -0.033648692, -0.014518415, 0.009318875, 0.021022039, -0.023641398, -0.043678377, -0.0037191545, 0.037476987, 0.008451352, -0.0045447005, -0.012111738, -0.024447354, -0.038999353, -0.026439859, -0.013466193, -0.036850132, 

### Procesamiento de Chunks Sliding Window

Ahora procesaremos los chunks de sliding window (620 chunks) para:
1. **Cargar** el archivo `data/chunks_sliding_v1.parquet`
2. **Generar embeddings** para cada chunk usando OpenAI
3. **Crear índice FAISS** (IndexFlatIP para cosine similarity)
4. **Guardar resultados**:
   - Índice FAISS: `vectordb/faiss_index_sliding.bin`
   - Metadata JSON: `vectordb/faiss_index_sliding_metadata.json`
   - Backup con embeddings: `data/embeddings_sliding_v1.parquet`

**Tiempo estimado**: ~30 segundos para 620 chunks (batch de 100)

In [18]:
import faiss
import json
import os
import pandas as pd

# ============================================================================
# 1. CARGAR CHUNKS DE RECURSIVE
# ============================================================================
print("=" * 80)
print("PROCESANDO CHUNKS RECURSIVE")
print("=" * 80)

ruta_chunks_recursive = os.path.join("data", "chunks_recursive_v1.parquet")

try:
    df_recursive = pd.read_parquet(ruta_chunks_recursive, engine="fastparquet")
    print(f" Chunks cargados: {len(df_recursive)} registros")
    print(f"   Columnas: {list(df_recursive.columns)}")
except Exception as e:
    print(f" Error al cargar chunks: {e}")
    raise

# Extraer textos de la columna 'chunk_text' (nota: diferente nombre que sliding)
texts_recursive = df_recursive["chunk_text"].tolist()
print(f" Textos a procesar: {len(texts_recursive)}")

# ============================================================================
# 2. GENERAR EMBEDDINGS
# ============================================================================
print("\n" + "=" * 80)
print("GENERANDO EMBEDDINGS")
print("=" * 80)

embeddings_recursive = generate_embeddings_batch(
    texts=texts_recursive,
    client=client,
    model="text-embedding-3-small",
    batch_size=100,
    normalize=True,
    show_progress=True
)

print(f"\n Embeddings generados: {embeddings_recursive.shape}")

# ============================================================================
# 3. CREAR ÍNDICE FAISS
# ============================================================================
print("\n" + "=" * 80)
print("CREANDO ÍNDICE FAISS")
print("=" * 80)

# Dimensión de los embeddings (1536 para text-embedding-3-small)
dimension = embeddings_recursive.shape[1]

# Crear índice FAISS con producto interno
index_recursive = faiss.IndexFlatIP(dimension)

# Agregar vectores al índice
index_recursive.add(embeddings_recursive)

print(f" Índice FAISS creado")
print(f"   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)")
print(f"   Dimensión: {index_recursive.d}")
print(f"   Vectores almacenados: {index_recursive.ntotal}")

# ============================================================================
# 4. GUARDAR ÍNDICE FAISS Y METADATA
# ============================================================================
print("\n" + "=" * 80)
print("GUARDANDO ÍNDICE Y METADATA")
print("=" * 80)

# Crear directorio vectordb (ya existe, pero por si acaso)
os.makedirs("vectordb", exist_ok=True)

# Guardar índice FAISS
index_path = os.path.join("vectordb", "faiss_index_recursive.bin")
faiss.write_index(index_recursive, index_path)
print(f" Índice guardado: {index_path}")

# Guardar metadata
metadata_recursive = {
    "index_type": "IndexFlatIP",
    "dimension": dimension,
    "total_vectors": int(index_recursive.ntotal),
    "model": "text-embedding-3-small",
    "chunking_method": "recursive text splitter",
    "chunk_size": 1000,
    "overlap": 200,
    "separators": ["\n\n", "\n", ". ", "; ", ", ", " "],
    "normalized": True,
    "columns": list(df_recursive.columns),
    "created_at": pd.Timestamp.now().isoformat()
}

metadata_path = os.path.join("vectordb", "faiss_index_recursive_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_recursive, f, indent=2, ensure_ascii=False)
print(f" Metadata guardada: {metadata_path}")

# ============================================================================
# 5. GUARDAR BACKUP CON EMBEDDINGS
# ============================================================================
print("\n" + "=" * 80)
print("GUARDANDO BACKUP CON EMBEDDINGS")
print("=" * 80)

# Agregar embeddings al DataFrame como nueva columna
df_recursive_with_embeddings = df_recursive.copy()
df_recursive_with_embeddings["embedding"] = list(embeddings_recursive)

# Guardar con fallback
backup_path = os.path.join("data", "embeddings_recursive_v1.parquet")
try:
    df_recursive_with_embeddings.to_parquet(backup_path, engine="fastparquet", index=False)
    print(f" Backup guardado: {backup_path}")
except Exception as e:
    print(f"  Error con parquet: {e}")
    backup_path_pkl = os.path.join("data", "embeddings_recursive_v1.pkl")
    df_recursive_with_embeddings.to_pickle(backup_path_pkl)
    print(f" Backup guardado (pickle): {backup_path_pkl}")

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "=" * 80)
print(" PROCESAMIENTO COMPLETADO: CHUNKS RECURSIVE")
print("=" * 80)
print(f" Chunks procesados: {len(df_recursive)}")
print(f" Embeddings generados: {embeddings_recursive.shape}")
print(f"  Índice FAISS: {index_recursive.ntotal} vectores")
print(f"\n Archivos generados:")
print(f"   - {index_path}")
print(f"   - {metadata_path}")
print(f"   - {backup_path if 'backup_path' in locals() else backup_path_pkl}")
print("=" * 80)

# ============================================================================
# COMPARACIÓN DE AMBOS ÍNDICES
# ============================================================================
print("\n" + "=" * 80)
print(" COMPARACIÓN DE ÍNDICES FAISS")
print("=" * 80)
print(f"Índice Sliding Window:")
print(f"   - Vectores: {index_sliding.ntotal}")
print(f"   - Método: Fixed-size sliding window")
print(f"   - Archivo: vectordb/faiss_index_sliding.bin")
print(f"\nÍndice Recursive:")
print(f"   - Vectores: {index_recursive.ntotal}")
print(f"   - Método: Recursive text splitter")
print(f"   - Archivo: vectordb/faiss_index_recursive.bin")
print(f"\n Total vectores almacenados: {index_sliding.ntotal + index_recursive.ntotal}")
print("=" * 80)

PROCESANDO CHUNKS RECURSIVE
 Chunks cargados: 988 registros
   Columnas: ['fuente', 'semana', 'fecha', 'apunte', 'titulo', 'autor', 'chunk_id', 'chunk_text', 'chunk_len']
 Textos a procesar: 988

GENERANDO EMBEDDINGS
Generando embeddings para 988 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí


Generando embeddings:   0%|          | 0/10 [00:00<?, ?batch/s]

Generando embeddings: 100%|██████████| 10/10 [00:13<00:00,  1.30s/batch]

Embeddings generados: shape (988, 1536)
   Rango de valores: [-0.1614, 0.1802]

 Embeddings generados: (988, 1536)

CREANDO ÍNDICE FAISS
 Índice FAISS creado
   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)
   Dimensión: 1536
   Vectores almacenados: 988

GUARDANDO ÍNDICE Y METADATA
 Índice guardado: vectordb\faiss_index_recursive.bin
 Metadata guardada: vectordb\faiss_index_recursive_metadata.json

GUARDANDO BACKUP CON EMBEDDINGS
  Error con parquet: Can't infer object conversion type: 0      [-0.026252314, 0.00065778283, 0.024926212, -0.04061467, 0.009046709, -0.0057370747, 0.018127132, 0.04169353, -0.06369782, -0.018261991, -0.00071432476, -0.0027421082, -0.05502197, 0.003452921, -0.031961292, -0.020352285, 0.0021872246, -0.022431344, -0.00081898004, 0.027960513, -0.040187616, -0.015564835, 0.0062484103, 0.018857613, -0.022352677, -0.04131143, -0.005192024, 0.03683865, 0.004363211, -0.004096305, -0.011889961, -0.025465643, -0.039648186, -0.02688165, -0.010299763, -0.03344473

### Demo de Búsqueda Semántica

Esta celda demuestra cómo usar los índices FAISS creados para búsqueda semántica:

**Flujo**:
1. Convertir consulta del usuario a embedding (mismo modelo)
2. Buscar top-k chunks más similares en ambos índices
3. Mostrar resultados con metadata (autor, semana, chunk, score)

**Ejemplo de uso**:
- "¿Qué es el aprendizaje por refuerzo?"
- "Explicación de redes neuronales convolucionales"
- "Métodos de optimización en deep learning"

**Métricas**:
- Score: Producto interno (0-1, mayor = más similar)
- Con vectores normalizados equivale a cosine similarity

In [19]:
def search_semantic(
    query: str,
    index_sliding,
    index_recursive,
    df_sliding,
    df_recursive,
    client,
    k: int = 5,
    model: str = "text-embedding-3-small"
):
    """
    Realiza búsqueda semántica en ambos índices FAISS.
    
    Args:
        query: Pregunta o texto de búsqueda
        index_sliding: Índice FAISS de sliding window
        index_recursive: Índice FAISS de recursive
        df_sliding: DataFrame con metadata de sliding
        df_recursive: DataFrame con metadata de recursive
        client: Cliente OpenAI
        k: Cantidad de resultados por índice (default: 5)
        model: Modelo de embeddings (debe coincidir con el usado en indexación)
        
    Returns:
        dict con resultados de ambos índices
    """
    print(f"\n{'='*80}")
    print(f" BÚSQUEDA SEMÁNTICA")
    print(f"{'='*80}")
    print(f"Query: \"{query}\"")
    print(f"Top-k por índice: {k}")
    
    # 1. Convertir query a embedding
    print(f"\n⏳ Generando embedding de la query...")
    query_embedding = generate_embeddings_batch(
        texts=[query],
        client=client,
        model=model,
        normalize=True,
        show_progress=False
    )
    
    resultados = {}
    
    # 2. Buscar en índice SLIDING WINDOW
    print(f"\n{'='*80}")
    print(f"RESULTADOS: SLIDING WINDOW")
    print(f"{'='*80}")
    
    distances_sliding, indices_sliding = index_sliding.search(query_embedding, k)
    
    resultados_sliding = []
    for i, (idx, score) in enumerate(zip(indices_sliding[0], distances_sliding[0]), 1):
        if idx == -1:  # FAISS devuelve -1 si no hay suficientes resultados
            continue
            
        row = df_sliding.iloc[idx]
        resultado = {
            "rank": i,
            "score": float(score),
            "fuente": row.get("fuente", "N/A"),
            "autor": row.get("autor", "N/A"),
            "semana": row.get("semana", "N/A"),
            "chunk": row.get("chunk", "")[:200] + "..." if len(str(row.get("chunk", ""))) > 200 else row.get("chunk", "")
        }
        resultados_sliding.append(resultado)
        
        print(f"\n{i}. Score: {score:.4f}")
        print(f"   Fuente: {resultado['fuente']}")
        print(f"   Autor: {resultado['autor']}")
        print(f"   Semana: {resultado['semana']}")
        print(f"   Chunk preview: {resultado['chunk']}")
    
    resultados["sliding_window"] = resultados_sliding
    
    # 3. Buscar en índice RECURSIVE
    print(f"\n{'='*80}")
    print(f" RESULTADOS: RECURSIVE")
    print(f"{'='*80}")
    
    distances_recursive, indices_recursive = index_recursive.search(query_embedding, k)
    
    resultados_recursive = []
    for i, (idx, score) in enumerate(zip(indices_recursive[0], distances_recursive[0]), 1):
        if idx == -1:
            continue
            
        row = df_recursive.iloc[idx]
        resultado = {
            "rank": i,
            "score": float(score),
            "fuente": row.get("fuente", "N/A"),
            "autor": row.get("autor", "N/A"),
            "semana": row.get("semana", "N/A"),
            "chunk": row.get("chunk_text", "")[:200] + "..." if len(str(row.get("chunk_text", ""))) > 200 else row.get("chunk_text", "")
        }
        resultados_recursive.append(resultado)
        
        print(f"\n{i}. Score: {score:.4f}")
        print(f"   Fuente: {resultado['fuente']}")
        print(f"   Autor: {resultado['autor']}")
        print(f"   Semana: {resultado['semana']}")
        print(f"   Chunk preview: {resultado['chunk']}")
    
    resultados["recursive"] = resultados_recursive
    
    print(f"\n{'='*80}")
    
    return resultados


# ============================================================================
# PRUEBA DE BÚSQUEDA SEMÁNTICA
# ============================================================================

# Ejemplo 1: Pregunta sobre conceptos de IA
print("\n" + "="*80)
print(" PRUEBA 1: Concepto de Inteligencia Artificial")
print("="*80)

resultados_1 = search_semantic(
    query="¿Qué es el aprendizaje por refuerzo?",
    index_sliding=index_sliding,
    index_recursive=index_recursive,
    df_sliding=df_sliding,
    df_recursive=df_recursive,
    client=client,
    k=3
)

# Ejemplo 2: Pregunta sobre redes neuronales
print("\n\n" + "="*80)
print("PRUEBA 2: Redes Neuronales")
print("="*80)

resultados_2 = search_semantic(
    query="Explicación de redes neuronales convolucionales",
    index_sliding=index_sliding,
    index_recursive=index_recursive,
    df_sliding=df_sliding,
    df_recursive=df_recursive,
    client=client,
    k=3
)

# Ejemplo 3: Consulta personalizada (cambia el texto aquí)
print("\n\n" + "="*80)
print(" PRUEBA 3: Consulta Personalizada")
print("="*80)

resultados_3 = search_semantic(
    query="métodos de optimización en machine learning",
    index_sliding=index_sliding,
    index_recursive=index_recursive,
    df_sliding=df_sliding,
    df_recursive=df_recursive,
    client=client,
    k=3
)

print("\n" + "="*80)
print(" DEMO COMPLETADA")
print("="*80)
print(" Tip: Modifica el parámetro 'query' en la última prueba para hacer tus propias búsquedas")
print("="*80)


 PRUEBA 1: Concepto de Inteligencia Artificial

 BÚSQUEDA SEMÁNTICA
Query: "¿Qué es el aprendizaje por refuerzo?"
Top-k por índice: 3

⏳ Generando embedding de la query...
Generando embeddings para 1 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí
Embeddings generados: shape (1, 1536)
   Rango de valores: [-0.0908, 0.1089]

RESULTADOS: SLIDING WINDOW

1. Score: 0.6545
   Fuente: 2_semana_ai_20250812_3.pdf
   Autor: None
   Semana: 2.0
   Chunk preview: enguaje, donde el sistema predice a partir de palabras previas cual es la palabra que sigue a dicha palabra, puede visualizarse la palabra previa como dato, pero a la vez como una misma etiqueta para ...

2. Score: 0.4825
   Fuente: 1_semana_ai_20250807_2.pdf
   Autor: fernando daniel brenes reyes
   Semana: 1.0
   Chunk preview: nden interactuando con un entorno y recibi- endo recompensas. aplicaciones: juegos (atari, go), control robotico. referencia clasica: sutton & barto. fig. 4. ejemplo de apren

## 📋 Resumen del Proyecto: Base de Datos Vectorial

### ✅ Archivos Generados

**Datos Procesados** (`data/`):
- `apuntes_clean_v1.parquet` - Dataset limpio con metadata (32 documentos)
- `chunks_sliding_v1.parquet` - Chunks de sliding window (620 chunks)
- `chunks_recursive_v1.parquet` - Chunks recursive (988 chunks)
- `embeddings_sliding_v1.parquet` - Backup con embeddings de sliding
- `embeddings_recursive_v1.parquet` - Backup con embeddings de recursive

**Base de Datos Vectorial** (`vectordb/`):
- `faiss_index_sliding.bin` - Índice FAISS para sliding window (620 vectores)
- `faiss_index_sliding_metadata.json` - Metadata del índice sliding
- `faiss_index_recursive.bin` - Índice FAISS para recursive (988 vectores)
- `faiss_index_recursive_metadata.json` - Metadata del índice recursive

**Total**: 1,608 vectores indexados para búsqueda semántica

---

### 🎯 Integración con RAG (Retrieval-Augmented Generation)

Para usar estos índices en un sistema RAG:

```python
import faiss
import pandas as pd
from openai import OpenAI

# 1. Cargar índice y metadata
index = faiss.read_index("vectordb/faiss_index_sliding.bin")
df_chunks = pd.read_parquet("data/chunks_sliding_v1.parquet")
client = OpenAI(api_key="tu-api-key")

# 2. Convertir pregunta del usuario a embedding
query = "¿Qué es el aprendizaje profundo?"
query_emb = client.embeddings.create(
    input=[query],
    model="text-embedding-3-small"
).data[0].embedding

# Normalizar
import numpy as np
query_emb = np.array([query_emb], dtype=np.float32)
query_emb = query_emb / np.linalg.norm(query_emb)

# 3. Buscar top-k chunks más relevantes
k = 5
distances, indices = index.search(query_emb, k)

# 4. Recuperar contexto
contexto = []
for idx in indices[0]:
    chunk_data = df_chunks.iloc[idx]
    contexto.append({
        "texto": chunk_data["chunk"],
        "fuente": chunk_data["fuente"],
        "autor": chunk_data["autor"]
    })

# 5. Construir prompt con contexto
contexto_texto = "\n\n".join([c["texto"] for c in contexto])
prompt = f"""Basándote en el siguiente contexto de los apuntes:

{contexto_texto}

Responde la pregunta: {query}"""

# 6. Generar respuesta con LLM
respuesta = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}]
)

print(respuesta.choices[0].message.content)
```

---

### 🔄 Comparación: Sliding Window vs Recursive

| Característica | Sliding Window | Recursive |
|----------------|----------------|-----------|
| **Chunks totales** | 620 | 988 |
| **Método** | Ventana fija (1000 chars, overlap 200) | Separadores semánticos |
| **Ventaja** | Cobertura uniforme, sin perder texto | Chunks más naturales (respeta párrafos) |
| **Uso recomendado** | Búsqueda exhaustiva | Búsqueda semántica precisa |

**Recomendación**: Prueba ambos índices y compara resultados. Puedes combinarlos (hybrid retrieval) para mayor robustez.

---

### 📊 Especificaciones Técnicas

- **Modelo de embeddings**: `text-embedding-3-small` (OpenAI)
- **Dimensiones**: 1536
- **Índice FAISS**: IndexFlatIP (producto interno ≈ cosine similarity con vectores normalizados)
- **Normalización**: L2 (vectores unitarios)
- **Costo estimado**: ~$0.03 USD para 1,608 chunks (~50,000 tokens)

---

### 🚀 Próximos Pasos

1. **Evaluar calidad de retrieval**: Hacer queries de prueba y validar relevancia de resultados
2. **Optimizar índice**: Si crece mucho, considerar IndexIVFFlat (clustering) para mayor velocidad
3. **Integrar con LLM**: Construir sistema RAG completo con GPT-4 o similar
4. **Agregar filtros**: Filtrar por autor, semana, o fuente antes de búsqueda
5. **Métricas**: Implementar evaluación con métricas como MRR, NDCG, o Recall@k

---

### 💡 Uso del Demo

La celda anterior contiene 3 ejemplos de búsqueda:
1. Aprendizaje por refuerzo
2. Redes neuronales convolucionales
3. Métodos de optimización (personalizable)

**Para hacer tu propia búsqueda**: Edita el parámetro `query` en la Prueba 3 y ejecuta la celda nuevamente.

---

¡Tu base de datos vectorial está lista para ser integrada en un sistema RAG! 🎉

### Procesamiento de Chunks Recursive

Ahora procesaremos los chunks recursive (988 chunks) con el mismo pipeline:
1. **Cargar** el archivo `data/chunks_recursive_v1.parquet`
2. **Generar embeddings** para cada chunk usando OpenAI
3. **Crear índice FAISS** (IndexFlatIP para cosine similarity)
4. **Guardar resultados**:
   - Índice FAISS: `vectordb/faiss_index_recursive.bin`
   - Metadata JSON: `vectordb/faiss_index_recursive_metadata.json`
   - Backup con embeddings: `data/embeddings_recursive_v1.parquet`

**Tiempo estimado**: ~50 segundos para 988 chunks (batch de 100)

**Nota**: Estos chunks son más "semánticos" que los de sliding window, ya que respetan límites naturales del texto (párrafos, oraciones, etc.).